## Import packages

In [56]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import xgboost as xgb
from xgboost import plot_importance

# Comparing Models for Daily Rat Sightings in New York City

Here are some things to do with thisnote book if I want to make this cleaner.

a) Delete the lines "rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])" at various places. We never use this column anyway.

b) Make sure to set rs['ds'] to date time when importing the data to begin with.

c) Make each section more robust having it reimport the data. That way, I can run each section individually.

## Importing the Data

In [57]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# This is should be 2251 which equals the number of days from 2020-01-01 to 2026-02-28
print(len(rs)==2251)


True


## Baseline Seasonal Average Model

In [58]:
years_back_use = 4
day_window_use = 4

In [59]:
def seasonal_average_forecast(data, target_dates, years_back=years_back_use, day_window=day_window_use):
    df = data.copy()
    df["ds"] = pd.to_datetime(df["ds"])
    df["doy"] = df["ds"].dt.dayofyear
    df["year"] = df["ds"].dt.year

    forecasts = []
    for target_date in target_dates:
        target_doy = target_date.dayofyear
        target_year = target_date.year
        mask = ((df["year"] >= target_year - years_back) & (df["year"] < target_year) & (np.abs(df["doy"] - target_doy) <= day_window))
        forecasts.append(df.loc[mask, "y"].mean())
    return pd.Series(forecasts, index=target_dates)

In [60]:
results = []
rs["ds"] = pd.to_datetime(rs["ds"])

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    
    train = rs.iloc[train_index].copy()
    test = rs.iloc[test_index].copy()
    
    # Target dates = the dates we want to forecast. There are 14 days.
    target_dates = test["ds"]
    
    # Seasonal forecast using only the training data (we will go back 5 years and take the average and use a day_window of 5 as well.)
    y_pred = seasonal_average_forecast(data=train, target_dates=target_dates, years_back=years_back_use,day_window=day_window_use)

    # We take the true values.
    y_true = test["y"].values
    
    # Compute the metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append the results of the metrics to the table as well as the fold number.
    results.append({"fold": i, "rmse": rmse, "mape": mape})

# Convert the data to a table for readability.
baseline_results_df = pd.DataFrame(results)

# We also include a new row which consists of the average RMSE and MAPE over each fold.
baseline_results_df.loc["mean"] = ["mean", baseline_results_df["rmse"].mean(), baseline_results_df["mape"].mean()]

baseline_results_df

,fold,rmse,mape
0,0,15.307003,0.285977
1,1,11.914386,0.186347
2,2,14.988228,0.250230
3,3,20.818662,0.348457
4,4,17.563835,0.256820
5,5,23.244116,0.361006
6,6,19.488233,0.266129
7,7,21.222927,0.284166
8,8,20.494910,0.301036
9,9,21.193296,0.277230


## Year Ago Rolling 4 Week Average 

In [61]:
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

## Just saving a copy for later
rs_saved = rs.copy()

In [62]:
# Tired of writing np.sqrt or typing a long name.
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

results = []

for fold, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]

    # Calculate the 4-week rolling average for the training data
    train_sorted = train.sort_values('ds') # making sure to sort it by date
    train_sorted['rolling_4w'] = train_sorted['y'].rolling(window=4, min_periods=1).mean()

    # This part of the code makes the predictions. We use the 'rolling_4w' column of the training set.
    y_pred = []
    y_true = test['y'].values

    for idx, row in test.iterrows():
        # Predict using the latest rolling average from the train data
        prediction = train_sorted['rolling_4w'].iloc[-1]  # Last value in the train rolling avg
        y_pred.append(prediction)
        
    # Calculate RMSE and MAPE for this fold
    fold_rmse = rmse(y_true, y_pred)
    fold_mape = mean_absolute_percentage_error(y_true, y_pred)
    
    results.append({'fold': fold, 'rmse': fold_rmse, 'mape': fold_mape})

rolling4w_results_df = pd.DataFrame(results)

overall_rmse = rolling4w_results_df['rmse'].mean()
overall_mape = rolling4w_results_df['mape'].mean()
rolling4w_results_df.loc['mean'] = ['mean', overall_rmse, overall_mape]

In [63]:
rolling4w_results_df

,fold,rmse,mape
0,0,14.323744,0.239272
1,1,12.719150,0.196755
2,2,12.398157,0.196394
3,3,19.240721,0.260374
4,4,15.923646,0.220183
5,5,17.378918,0.256428
6,6,24.564601,0.247419
7,7,20.989793,0.255484
8,8,19.858428,0.253080
9,9,18.798509,0.187194


## Prophet Model

In [64]:
date_range = pd.date_range(start="2020-01-01", end="2026-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

In [65]:
# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

prophet_results_df = pd.DataFrame(results)
prophet_results_df.loc['mean'] = ['mean',  prophet_results_df['rmse'].mean(), prophet_results_df['mape'].mean()]
prophet_results_df

22:13:28 - cmdstanpy - INFO - Chain [1] start processing
22:13:28 - cmdstanpy - INFO - Chain [1] done processing
22:13:29 - cmdstanpy - INFO - Chain [1] start processing
22:13:29 - cmdstanpy - INFO - Chain [1] done processing
22:13:30 - cmdstanpy - INFO - Chain [1] start processing
22:13:31 - cmdstanpy - INFO - Chain [1] done processing
22:13:31 - cmdstanpy - INFO - Chain [1] start processing
22:13:32 - cmdstanpy - INFO - Chain [1] done processing
22:13:32 - cmdstanpy - INFO - Chain [1] start processing
22:13:33 - cmdstanpy - INFO - Chain [1] done processing
22:13:34 - cmdstanpy - INFO - Chain [1] start processing
22:13:34 - cmdstanpy - INFO - Chain [1] done processing
22:13:35 - cmdstanpy - INFO - Chain [1] start processing
22:13:35 - cmdstanpy - INFO - Chain [1] done processing
22:13:36 - cmdstanpy - INFO - Chain [1] start processing
22:13:36 - cmdstanpy - INFO - Chain [1] done processing
22:13:37 - cmdstanpy - INFO - Chain [1] start processing
22:13:37 - cmdstanpy - INFO - Chain [1]

,fold,rmse,mape
0,0,12.928150,0.245960
1,1,9.293687,0.142832
2,2,7.988090,0.130246
3,3,9.851386,0.143899
4,4,10.863706,0.137638
5,5,15.435334,0.215371
6,6,11.555594,0.145927
7,7,11.307567,0.135042
8,8,10.504938,0.117917
9,9,8.858023,0.110440


## NeuralProphet Model

Since the Prophet model seemed to work so well (see below), we might ask about whether or not one cna improve on it. We try the ['Neural Prophet model'](https://neuralprophet.com/) which should theoretically provide the either the same results or improvements.

In [66]:
from neuralprophet import NeuralProphet

import numpy as np
np.NaN = np.nan


# the following packages are meant to turn off a bunch of the warnings and ERRORs that pop up while running NeuralProphet.
# the errors that do show up are not all that important and a lot is due to outdated packages.
import warnings
import logging

warnings.filterwarnings("ignore")

logging.getLogger("neuralprophet").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("NP").setLevel(logging.ERROR)

In [67]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# This is should be 2251 which equals the number of days from 2020-01-01 to 2026-02-28
print(len(rs)==2251)


True


In [68]:
## Add weather data.

import requests
import pandas as pd

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

def add_weather_data_no_index(df,wd):
    if "time" in wd.columns:
        wd = wd.drop(columns=["time"])

    for column in wd.columns:
        df[column] = wd[column].values

    return df

In [69]:
regressed_features = ['apparent_temperature_max', 
                      'apparent_temperature_min',
                    'snowfall_sum']
wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

In [70]:
lags_for_regressed_features = dict()
lags_for_regressed_features['apparent_temperature_max'] = 30
lags_for_regressed_features['apparent_temperature_min'] = 14
lags_for_regressed_features['snowfall_sum'] = 3

In [71]:
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):

    train = rs.iloc[train_index].copy()
    test = rs.iloc[test_index].copy()

    train = train.dropna(subset=["y"])


    model = NeuralProphet(yearly_seasonality=True, 
                          weekly_seasonality=True, 
                          epochs = 200,
                          accelerator = 'auto',
                          n_lags=14)

    model = model.add_country_holidays(country_name="US")

    for column in regressed_features:
        model.add_lagged_regressor(column, n_lags=lags_for_regressed_features[column])

    # merge regressors correctly
    # train = train.merge(wd[['ds'] + regressed_features], on="ds", how="left")

    model.fit(train, freq="D", progress="off")

    # build dataframe containing future regressors
    future = pd.concat([train[['ds','y'] + regressed_features], test[['ds','y']].merge(wd[['ds'] + regressed_features], on="ds", how="left")])
    forecast = model.predict(future)

    y_pred = forecast["yhat1"].iloc[-len(test):].values
    y_true = test["y"].values

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    results.append({"fold": i, "rmse": rmse, "mape": mape})

neural_prophet_results_df = pd.DataFrame(results)
neural_prophet_results_df.loc["mean"] = ["mean", neural_prophet_results_df["rmse"].mean(), neural_prophet_results_df["mape"].mean()]
neural_prophet_results_df

Finding best initial lr:   0%|          | 0/232 [00:00<?, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 59it [00:00, ?it/s]

Finding best initial lr:   0%|          | 0/232 [00:00<?, ?it/s]

Training: 0it [00:00, ?it/s]

...

Predicting: 35it [00:00, ?it/s]

Finding best initial lr:   0%|          | 0/234 [00:00<?, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 35it [00:00, ?it/s]

,fold,rmse,mape
0,0,13.767870,0.229247
1,1,9.628497,0.114966
2,2,8.468593,0.142349
3,3,10.982896,0.161305
4,4,12.371418,0.159688
5,5,11.030414,0.141677
6,6,12.656647,0.157797
7,7,13.416199,0.166526
8,8,12.015590,0.163001
9,9,12.227787,0.129494


### Plots for Neural Prophet

To see if tuning the parameters might help and to see the effects of the lagged regressors, we use NeuralProphet's built in plotting.

In [72]:
model.plot(forecast)

ERROR - (NP.plotly.plot) - plotly-resampler is not installed. Please install it to use the resampler.


In [73]:
model.plot_parameters()

ERROR - (NP.plotly.plot_parameters) - plotly-resampler is not installed. Please install it to use the resampler.


## SARIMAX Model

The reason why the SARIMA / SARIMAX model does not perform as well as we'd like is discussed here: https://stats.stackexchange.com/questions/613677/using-sarimax-for-daily-data-with-yearly-seasonal-pattern. An excellent read for more details can be found here: https://robjhyndman.com/hyndsight/longseasonality/. For these reasons, instead of using SARIMA's included seasonality features, we instead add Fourier terms as exogeneous variables. We will only add Fourier terms to capture yearly seasonality for now.

Furthermore, we make the point that ARIMA is only good over a short interval. Since our goal is 14 days out, as opposed to 7 days, the predictions are not as good. We also run into the issue that there are multiple seasonalities in the rat sightings data.

In [74]:
pip install pmdarima

Note: you may need to restart the kernel to use updated packages.


In [75]:
from pmdarima import auto_arima

In [76]:
def fourier_terms(df, periods, n_terms):
    t = np.arange(1, len(df) + 1)
    fourier_df = pd.DataFrame()
    
    for period in periods:
        for i in range(1, n_terms + 1):
            fourier_df[f'{period}sin_{i}'] = np.sin(2 * np.pi * i * t / period)
            fourier_df[f'{period}cos_{i}'] = np.cos(2 * np.pi * i * t / period)
    
    return fourier_df

fourier_train = fourier_terms(rs, [365], 10)
exog = fourier_train

In [77]:
exog

,365sin_1,365cos_1,365sin_2,365cos_2,365sin_3,365cos_3,365sin_4,365cos_4,365sin_5,365cos_5,365sin_6,365cos_6,365sin_7,365cos_7,365sin_8,365cos_8,365sin_9,365cos_9,365sin_10,365cos_10
0,0.017213,0.999852,0.034422,0.999407,0.051620,0.998667,0.068802,0.997630,0.085965,0.996298,0.103102,0.994671,0.120208,0.992749,0.137279,0.990532,0.154309,0.988023,0.171293,0.985220
1,0.034422,0.999407,0.068802,0.997630,0.103102,0.994671,0.137279,0.990532,0.171293,0.985220,0.205104,0.978740,0.238673,0.971100,0.271958,0.962309,0.304921,0.952378,0.337523,0.941317
2,0.051620,0.998667,0.103102,0.994671,0.154309,0.988023,0.205104,0.978740,0.255353,0.966848,0.304921,0.952378,0.353676,0.935368,0.401488,0.915864,0.448229,0.893919,0.493776,0.869589
3,0.068802,0.997630,0.137279,0.990532,0.205104,0.978740,0.271958,0.962309,0.337523,0.941317,0.401488,0.915864,0.463550,0.886071,0.523416,0.852078,0.580800,0.814046,0.635432,0.772157
4,0.085965,0.996298,0.171293,0.985220,0.255353,0.966848,0.337523,0.941317,0.417194,0.908818,0.493776,0.869589,0.566702,0.823923,0.635432,0.772157,0.699458,0.714673,0.758306,0.651899
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2246,0.831171,0.556017,0.924291,-0.381689,0.196673,-0.980469,-0.705584,-0.708627,-0.981306,0.192452,-0.385663,0.922640,0.552435,0.833556,0.999991,0.004304,0.559589,-0.828770,-0.377708,-0.925925
2247,0.840618,0.541628,0.910605,-0.413279,0.145799,-0.989314,-0.752667,-0.658402,-0.961130,0.276097,-0.288482,0.957485,0.648630,0.761104,0.991114,-0.133015,0.425000,-0.905193,-0.530730,-0.847541
2248,0.849817,0.527078,0.895839,-0.444378,0.094537,-0.995521,-0.796183,-0.605056,-0.933837,0.357698,-0.188227,0.982126,0.735417,0.677615,0.963471,-0.267814,0.280231,-0.959933,-0.668064,-0.744104
2249,0.858764,0.512371,0.880012,-0.474951,0.043022,-0.999074,-0.835925,-0.548843,-0.899631,0.436651,-0.085965,0.996298,0.811539,0.584298,0.917584,-0.397543,0.128748,-0.991677,-0.785650,-0.618671


In [78]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Make sure the columns for SARIMA model are renamed.
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

results = []

# Loop through each fold
for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]

    exog_train = exog.iloc[train_index]
    exog_test = exog.iloc[test_index]

    orders = (1,1,1)
    seasonal_orders = (0,0,0,0)

    # we can use auto_arima to get optimal (p, d, q) and (P, D, Q, s) parameters for SARIMAX. just need to uncomment the following code.
    # model_auto = auto_arima(train['y'], 
    #                         exog=exog_train,  # exogenous Fourier terms for training data
    #                         seasonal=True, 
    #                         m=7, #  
    #                         trace=True, 
    #                         stepwise=True,  # Stepwise search to speed up
    #                         suppress_warnings=True, 
    #                         maxiter=300,  # Limit the number of iterations
    #                         max_p=3, 
    #                         max_q=3, 
    #                         max_P=2, 
    #                         max_Q=2, 
    #                         d=1,# might want to tune d 
    #                         D=1 # might want to tune D
    #                         )
    # orders = model_auto.order  # (p, d, q)
    # seasonal_orders = model_auto.seasonal_order  # (P, D, Q, s)
    
    # Fit the SARIMAX model with the exogenous features (Fourier terms)
    model_sarimax = SARIMAX(train['y'], 
                            order=orders,  
                            seasonal_order=seasonal_orders,  
                            exog=exog_train,  # Exogenous Fourier terms for training data
                            )
    
    model_fit = model_sarimax.fit(disp=False)
    
    # Predict for the test period. Have to remember to subtract 1 to get the correct index.
    y_pred = model_fit.predict(start=len(train), end=len(train)+len(test)-1, exog=exog_test, dynamic=False)
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

sarima_results_df = pd.DataFrame(results)

In [79]:
sarima_results_df.loc['mean'] = ['mean',  sarima_results_df['rmse'].mean(), sarima_results_df['mape'].mean()]

In [80]:
sarima_results_df

,fold,rmse,mape
0,0,15.174437,0.240537
1,1,15.002202,0.201508
2,2,12.875488,0.199047
3,3,17.310608,0.267932
4,4,14.457614,0.204142
5,5,17.378153,0.257355
6,6,14.693533,0.177519
7,7,20.289146,0.245472
8,8,20.150224,0.262065
9,9,14.655524,0.176261


## Holt-Winters Model

In [81]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

In [82]:
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []
for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    holt_winters = ExponentialSmoothing(train['y'], seasonal='add', seasonal_periods=365).fit(optimized=True)
    
    y_pred = holt_winters.forecast(len(test))
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

hw_results_df = pd.DataFrame(results)
hw_results_df.loc['mean'] = ['mean',  hw_results_df['rmse'].mean(), hw_results_df['mape'].mean()]
hw_results_df

,fold,rmse,mape
0,0,17.167550,0.280495
1,1,17.486979,0.255434
2,2,17.312305,0.297496
3,3,22.508395,0.356789
4,4,19.906609,0.267117
5,5,20.725266,0.310178
6,6,22.480081,0.270543
7,7,25.834222,0.311912
8,8,23.345596,0.306865
9,9,21.350550,0.274466


## MSTL Model

MSTL Model stands for Multiple Seasonal-Trend decomposition using LOESS. For more, see https://nixtlaverse.nixtla.io/statsforecast/docs/models/multipleseasonaltrend.html.

In [83]:
# !pip install statsforecast

In [84]:
# from statsforecast import StatsForecast
# from statsforecast.models import MSTL, AutoARIMA

In [85]:
# tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

# # again, we reimport the data for ease of running

# rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
# rs['created_date'] = pd.to_datetime(rs['created_date']) 
# rs['closed_date'] = pd.to_datetime(rs['closed_date'])
# rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# # mark cutoff dates, and also rename columns
# rs = rs[rs['created_date']<'2026-03-01']
# rs = rs[rs['created_date']>='2020-01-01']
# rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
# rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# # This is should be 2251 which equals the number of days from 2020-01-01 to 2026-02-28
# print(len(rs)==2251)


In [86]:
# # specific to Statsforecast requirements

# rs.columns = ['ds', 'y']
# rs.insert(0, 'unique_id', 'DAILY_RAT_SIGHTINGS')
# rs = rs.sort_values(['unique_id', 'ds']).reset_index(drop=True)
# rs.tail()

In [87]:
# rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# results = []

# def create_dataset(series, time_step=1):
#     X, y = [], []
#     for i in range(len(series) - time_step):
#         X.append(series[i:(i + time_step), 0])
#         y.append(series[i + time_step, 0])
#     return np.array(X), np.array(y)

# time_step = 10
# results = []

# for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    
#     train = rs.iloc[train_index]
#     test = rs.iloc[test_index]

#     series = train['y'].values

#     # scale the data
#     models = [MSTL(
#     season_length=[7, 7*4, 7*52], # seasonalities of the time series (weekly, monthly, yearly)
#     trend_forecaster=AutoARIMA() # model used to forecast trend
#     )]
#     sf = StatsForecast(
#     models=models, # model used to fit
#     freq='d', # frequency of the data
#     )

#     sf = sf.fit(df=rs)
#     forecasts = sf.predict(h=14, level=[90]) # 90 means (confidence percentile) of the prediction interval. 

#     y_pred = forecasts['MSTL'].values
#     y_true = test['y'].values

#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
#     mape = mean_absolute_percentage_error(y_true, y_pred)

#     results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# mstl_results_df = pd.DataFrame(results)
# mstl_results_df.loc['mean'] = ['mean',  mstl_results_df['rmse'].mean(), mstl_results_df['mape'].mean()]
# mstl_results_df

## LSTM Model

LSTM stands for Long Term Short Memory. Here, we simply try to use the LSTM model by itself and check how it performs. For future purposes, it might be of interest to see if a ['hybrid model'](https://peerj.com/articles/cs-1001/#proposed-hybrid-model) using Prophet and LSTM might be able to produce better results.

In [88]:
# pip install neuralforecast

In [89]:
# from neuralforecast import NeuralForecast
# from neuralforecast.models import LSTM
# import pandas as pd
# import numpy as np

In [90]:
# tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

# # again, we reimport the data for ease of running

# rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
# rs['created_date'] = pd.to_datetime(rs['created_date']) 
# rs['closed_date'] = pd.to_datetime(rs['closed_date'])
# rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# # mark cutoff dates, and also rename columns
# rs = rs[rs['created_date']<'2026-03-01']
# rs = rs[rs['created_date']>='2020-01-01']
# rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
# rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# # This is should be 2251 which equals the number of days from 2020-01-01 to 2026-02-28
# print(len(rs)==2251)

# rs['ds'] = pd.to_datetime(rs['ds'])
# rs['ds'] = rs['ds'].astype('datetime64[ns]')

In [91]:
# time_step = 14
# results = []

# for i, (train_index, test_index) in enumerate(tscv.split(rs)):

#     train = rs.iloc[train_index].copy()
#     test = rs.iloc[test_index].copy()

#     # need to do some heavy formatting
#     train_df = train[['ds', 'y']].copy()
#     train_df['unique_id'] = 'series_1'
#     train_df = train_df[['unique_id', 'ds', 'y']]
#     test_df = test[['ds', 'y']].copy()
#     test_df['unique_id'] = 'series_1'
#     test_df = test_df[['unique_id', 'ds', 'y']]

#     horizon = len(test_df)

#     models = [LSTM(h=horizon, max_steps=100, scaler_type='standard')]
#     nf = NeuralForecast(models=models, freq='D')

#     nf.fit(df=train_df)
#     forecasts = nf.predict(h=14)

#     y_pred = forecasts['LSTM'].values
#     y_true = test_df['y'].values

#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
#     mape = mean_absolute_percentage_error(y_true, y_pred)

#     results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# lstm_results_df = pd.DataFrame(results)
# lstm_results_df.loc['mean'] = ['mean',  lstm_results_df['rmse'].mean(), lstm_results_df['mape'].mean()]
# lstm_results_df

## XGBoost Model

The XGBoost model requires a bit more preparatory work. Our current dataframe rs is quite bare. We will need to add features for use.

In [92]:
import xgboost as xgb
from xgboost import plot_importance

### Adding Features to XGBoost

In [93]:
def create_features(df):
    # create time series features based on time series index.
    df = df.copy()
    df['dayofweek'] = df.index.dayofweek
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['dayofyear'] = df.index.dayofyear
    df['dayofmonth'] = df.index.day
    df['weekofyear'] = df.index.isocalendar().week
    return df

def add_cyclic(df):
    # features to handly cyclic behavior
    target_map = df['y'].to_dict()
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek']/7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek']/7)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
    return df

def add_lags(df):
    # lags
    target_map = df['y'].to_dict()
    df['lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
    df['lag2'] = (df.index - pd.Timedelta('2 days')).map(target_map)
    df['lag3'] = (df.index - pd.Timedelta('3 days')).map(target_map)
    df['lag4'] = (df.index - pd.Timedelta('4 days')).map(target_map)
    df['lag5'] = (df.index - pd.Timedelta('5 days')).map(target_map)
    df['lag6'] = (df.index - pd.Timedelta('6 days')).map(target_map)
    df['lag7'] = (df.index - pd.Timedelta('7 days')).map(target_map)
    df['lag8'] = (df.index - pd.Timedelta('8 days')).map(target_map)
    df['lag9'] = (df.index - pd.Timedelta('9 days')).map(target_map)
    df['lag10'] = (df.index - pd.Timedelta('10 days')).map(target_map)
    df['lag11'] = (df.index - pd.Timedelta('11 days')).map(target_map)
    df['lag12'] = (df.index - pd.Timedelta('12 days')).map(target_map)
    df['lag13'] = (df.index - pd.Timedelta('13 days')).map(target_map)
    df['lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    return df

def add_seasonal_lags(df):
    # lags of various lengths for different levels of seasonality
    target_map = df['y'].to_dict()
    df['lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)

    df['lag362'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag363'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag364'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag366'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag367'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    
    df['lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)
    df['lag1095'] = (df.index - pd.Timedelta('1095 days')).map(target_map)
    df['lag1460'] = (df.index - pd.Timedelta('1460 days')).map(target_map)
    df['lag1825'] = (df.index - pd.Timedelta('1825 days')).map(target_map)
    return df

def add_moving_averages(df):
    df = df.copy()
    df = df.sort_index()
    
    # Moving averages (using previous values only)
    df['ma7'] = df['y'].shift(1).rolling(window=7).mean()
    df['ma30'] = df['y'].shift(1).rolling(window=30).mean()
    df['ma60'] = df['y'].shift(1).rolling(window=60).mean()
    df['ma90'] = df['y'].shift(1).rolling(window=90).mean()
    df['ma120'] = df['y'].shift(1).rolling(window=120).mean()
    df['ma150'] = df['y'].shift(1).rolling(window=150).mean()
    df['ma180'] = df['y'].shift(1).rolling(window=180).mean()
    df['ma365'] = df['y'].shift(1).rolling(window=365).mean()
    
    return df


In the next two code block, we add weather data to the data set. This is not optimized i.e. we just obtain the weather data in Manhattan and hope that it is representative of the average weather over the whole city.

In [94]:
## Add weather data.

import requests
import pandas as pd

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [95]:
def add_weather_data(df, wd):
    df = df.copy()
    wd = wd.copy()
    
    # Ensure datetime index 
    df.index = pd.to_datetime(df.index)
    wd.index = pd.to_datetime(wd.index)
    
    # Drop unnecessary columns
    if "time" in wd.columns:
        wd = wd.drop(columns=["time"])
    
    # Join on date index
    df = df.join(wd, how="left")
    
    return df

In [96]:
from pandas.tseries.holiday import USFederalHolidayCalendar

def add_federal_holidays(df, custom_holidays=None):
    df = df.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df.index.min(), end=df.index.max())
    
    if custom_holidays:
        for d in custom_holidays:
            if len(d) == 5:  # MM-DD format handling
                years = df.index.year.unique()
                for y in years:
                    holidays = holidays.append(pd.to_datetime([f"{y}-{d}"]))
            else:  # YYYY-MM-DD format handling
                holidays = holidays.append(pd.to_datetime([d]))
    
    holidays = holidays.drop_duplicates().sort_values()
    
    df["is_federal_holiday"] = df.index.isin(holidays).astype(int)
    
    return df

In [97]:
def add_law_flag(df, law_name: str, start_date: str):
    # Adds a binary column to indicate when a new law is active.
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    start_dt = pd.to_datetime(start_date)
    # Create binary column: 1 if date >= start_date, else 0
    df[law_name] = (df.index >= start_dt).astype(int)
    
    return df

In [98]:
# This must be run after importing weather data

def add_more_weather_feature(df):
    target_map = df['apparent_temperature_min'].to_dict()
    df['apparent_temperature_min_lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
    df['apparent_temperature_min_lag7'] = (df.index - pd.Timedelta('7 days')).map(target_map)
    df['apparent_temperature_min_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['apparent_temperature_min_lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df['apparent_temperature_min_lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df['apparent_temperature_min_lag17'] = (df.index - pd.Timedelta('17 days')).map(target_map)
    df['apparent_temperature_min_lag18'] = (df.index - pd.Timedelta('18 days')).map(target_map)
    df['apparent_temperature_min_lag19'] = (df.index - pd.Timedelta('19 days')).map(target_map)
    df['apparent_temperature_min_lag20'] = (df.index - pd.Timedelta('20 days')).map(target_map)
    df['apparent_temperature_min_lag21'] = (df.index - pd.Timedelta('21 days')).map(target_map)

    df['apparent_temperature_min_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['apparent_temperature_min_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['apparent_temperature_min_lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['apparent_temperature_min_lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['apparent_temperature_min_lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['apparent_temperature_min_lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)
    df['apparent_temperature_min_lag210'] = (df.index - pd.Timedelta('210 days')).map(target_map)
    df['apparent_temperature_min_lag240'] = (df.index - pd.Timedelta('240 days')).map(target_map)
    df['apparent_temperature_min_lag270'] = (df.index - pd.Timedelta('270 days')).map(target_map)
    df['apparent_temperature_min_lag300'] = (df.index - pd.Timedelta('300 days')).map(target_map)
    df['apparent_temperature_min_lag330'] = (df.index - pd.Timedelta('330 days')).map(target_map)
    df['apparent_temperature_min_lag360'] = (df.index - pd.Timedelta('360 days')).map(target_map)
    df['apparent_temperature_min_lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['apparent_temperature_min_lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)

    target_map = df['temperature_2m_max'].to_dict()
    df['temperature_2m_max_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['temperature_2m_max_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['temperature_2m_max_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)

    return df

In [99]:
rs = rs.set_index('ds')
rs.index = pd.to_datetime(rs.index)

In [ ]:
rs = create_features(rs)
rs = add_cyclic(rs)
rs = add_lags(rs)
rs = add_seasonal_lags(rs)
rs = add_moving_averages(rs)
# rs = add_weather_data(rs,wd)
rs = add_more_weather_feature(rs)
rs = add_federal_holidays(rs, custom_holidays = ['12-31'])
rs = add_law_flag(rs, law_name='Trash_Law', start_date = '2024-03-01')
rs = add_law_flag(rs, law_name = 'New_Trash_Law', start_date = '2024-11-01')
rs = add_law_flag(rs, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
rs = add_law_flag(rs, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

ValueError: columns overlap but no suffix specified: Index(['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum'], dtype='object')

### Features for XGBoost

In [ ]:
# FEATURES = ['apparent_temperature_min_lag30',
#             'apparent_temperature_min_lag60',
#             'apparent_temperature_min_lag120',
#             'apparent_temperature_min_lag365',
#             'apparent_temperature_min_lag730',
#             'dayofyear', 
#             'temperature_2m_max_lag30',
#             'temperature_2m_max_lag60'
#             ]

### Add default parameters for XGBoost

In [ ]:
# params = {'objective': 'reg:squarederror',
#          'eval_metric': 'rmse',
#          'booster': 'gbtree',
#          'base_score': 0.5, 
#          'n_estimators': 1000, 
#         #  'min_child_weight': 6, 
#         # 'learning_rate': 0.01,
#         # 'max_depth': 8, 
#         #  'subsample': 1,
#         #  'colsample_bytree': 0.96,
#         #  'colsample_bylevel': 0.6, 
#         #  'colsample_bynode': 0.9, 
#         #  'reg_alpha': 2.2, 
#         #  'gamma': 100, 
#         #  'reg_lambda': 0.18,
#         #  'early_stopping_rounds': 100, 
#         }


### Results for XGBoost Model

In [ ]:
# print(FEATURES)
# print(params)
# TARGET = 'y'

# # Gotta make sure the features and parameters exist.

# reg = xgb.XGBRegressor(**params)
# results = []

# for i, (train_index, test_index) in enumerate(tscv.split(rs)):
#     train = rs.iloc[train_index]
#     test = rs.iloc[test_index]
    
#     reg.fit(train[FEATURES], train[TARGET])
#     y_pred = reg.predict(test[FEATURES])
#     y_true = test[TARGET].values
    
#     # Our metrics
#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
#     mape = mean_absolute_percentage_error(y_true, y_pred)
    
#     results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# xgb_results_df = pd.DataFrame(results)
# mean_rmse = xgb_results_df['rmse'].mean()
# mean_mape = xgb_results_df['mape'].mean()
# xgb_results_df.loc['mean'] = ['mean', mean_rmse, mean_mape]

['apparent_temperature_min_lag30', 'apparent_temperature_min_lag60', 'apparent_temperature_min_lag120', 'apparent_temperature_min_lag365', 'apparent_temperature_min_lag730', 'dayofyear', 'temperature_2m_max_lag30', 'temperature_2m_max_lag60']
{'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'booster': 'gbtree', 'base_score': 0.5, 'n_estimators': 1000}


KeyError: "['apparent_temperature_min_lag30', 'apparent_temperature_min_lag60', 'apparent_temperature_min_lag120', 'apparent_temperature_min_lag365', 'apparent_temperature_min_lag730', 'temperature_2m_max_lag30', 'temperature_2m_max_lag60'] not in index"

In [ ]:
# xgb_results_df

NameError: name 'xgb_results_df' is not defined

## XGBoosted Prophet Model

In [105]:
## Recall the copy that was saved. 
## This was because Prophet demanded a particular format while SARIMA and Holt-Winters didn't.
rs_saved
rs = rs_saved

In [106]:
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    # Split the dataset into training and testing sets
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    # Fit Prophet on the training data
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')
    model.fit(train)
    
    # Make predictions on the training set to calculate residuals
    train_future = model.make_future_dataframe(periods=0, freq='D')  # Use periods=0 to only use the training data
    train_forecast = model.predict(train_future)
    
    # Calculate residuals (actual - predicted) on the training data
    train_residuals = train['y'].values - train_forecast['yhat'][:len(train)].values
    
    # Build a new DataFrame of residuals
    residuals_df = pd.DataFrame({'ds': train['ds'], 'y': train_residuals })
    # Add the features I defined above
    train = train.set_index('ds')
    train.index = pd.to_datetime(train.index)
    train = create_features(train)
    train = add_cyclic(train)
    train = add_lags(train)
    train = add_seasonal_lags(train)
    train = add_moving_averages(train)
    train = add_weather_data(train,wd)
    train = add_more_weather_feature(train)
    train = add_federal_holidays(train, custom_holidays = ['12-31'])
    train = add_law_flag(train, law_name='Trash_Law', start_date = '2024-03-01')
    train = add_law_flag(train, law_name = 'New_Trash_Law', start_date = '2024-11-01')
    train = add_law_flag(train, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
    train = add_law_flag(train, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

    X_train_residuals = train[FEATURES]
    y_train_residuals = residuals_df['y']
    
    xgb_model = xgb.XGBRegressor(**params)
    xgb_model.fit(X_train_residuals, y_train_residuals)
    
    # Add the features I defined above (I should probably define a new function to do this in one go...)
    test = test.set_index('ds')
    test.index = pd.to_datetime(test.index)
    test = create_features(test)
    test = add_cyclic(test)
    test = add_lags(test)
    test = add_seasonal_lags(test)
    test = add_moving_averages(test)
    test = add_weather_data(test,wd)
    test = add_more_weather_feature(test)
    test = add_federal_holidays(test, custom_holidays = ['12-31'])
    test = add_law_flag(test, law_name='Trash_Law', start_date = '2024-03-01')
    test = add_law_flag(test, law_name = 'New_Trash_Law', start_date = '2024-11-01')
    test = add_law_flag(test, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
    test = add_law_flag(test, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

    # Predict residuals using XGBoost for the test set
    X_test = test[FEATURES]  # Features for the test set
    xgb_residual_preds = xgb_model.predict(X_test)
    
    # Forecast using Prophet on the test set
    future = model.make_future_dataframe(periods=len(test), freq='D')
    prophet_forecast = model.predict(future)
    
    # Combine Prophet's forecast and XGBoost's residual prediction
    y_pred = prophet_forecast['yhat'][-len(test):].values + xgb_residual_preds
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Store the results for this fold
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})
    
    
    ## Uncomment code below if you want to have plots on feature importance. I'll leave it commented out for obvious reasons.
    # fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(10, 30))
    # plot_importance(xgb_model, ax=ax1, importance_type='gain')
    # ax1.set_title('Gain-based Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax2, importance_type='weight')
    # ax2.set_title('Split-based Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax3, importance_type='cover')
    # ax3.set_title('Cover Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax4, importance_type='total_gain')
    # ax4.set_title('Total Gain Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax5, importance_type='total_cover')
    # ax5.set_title('Total Cover Importance', fontsize=12)

    plt.show()

# Convert the results into a DataFrame
prophet_xgb_results_df = pd.DataFrame(results)
mean_rmse = prophet_xgb_results_df['rmse'].mean()
mean_mape = prophet_xgb_results_df['mape'].mean()
prophet_xgb_results_df.loc['mean'] = ['mean', mean_rmse, mean_mape]

23:23:48 - cmdstanpy - INFO - Chain [1] start processing
23:23:48 - cmdstanpy - INFO - Chain [1] done processing
23:23:51 - cmdstanpy - INFO - Chain [1] start processing
23:23:52 - cmdstanpy - INFO - Chain [1] done processing
23:23:55 - cmdstanpy - INFO - Chain [1] start processing
23:23:55 - cmdstanpy - INFO - Chain [1] done processing
23:23:58 - cmdstanpy - INFO - Chain [1] start processing
23:23:58 - cmdstanpy - INFO - Chain [1] done processing
23:24:01 - cmdstanpy - INFO - Chain [1] start processing
23:24:01 - cmdstanpy - INFO - Chain [1] done processing
23:24:04 - cmdstanpy - INFO - Chain [1] start processing
23:24:04 - cmdstanpy - INFO - Chain [1] done processing
23:24:07 - cmdstanpy - INFO - Chain [1] start processing
23:24:07 - cmdstanpy - INFO - Chain [1] done processing
23:24:11 - cmdstanpy - INFO - Chain [1] start processing
23:24:11 - cmdstanpy - INFO - Chain [1] done processing
23:24:14 - cmdstanpy - INFO - Chain [1] start processing
23:24:14 - cmdstanpy - INFO - Chain [1]

In [107]:
prophet_xgb_results_df

,fold,rmse,mape
0,0,13.340758,0.253073
1,1,9.451640,0.145749
2,2,8.913333,0.134629
3,3,9.232191,0.131012
4,4,10.577410,0.129190
5,5,18.009354,0.267527
6,6,11.376798,0.143244
7,7,11.997262,0.137025
8,8,11.127300,0.123115
9,9,8.152440,0.102250


# Conclusions on Model Comparisons

## Results Table

In [108]:
# We make a dictionary of models and their results to make it easier to iterate over. 
# Make sure to add to this if writing new models in.
models = {
    'baseline': baseline_results_df,
    'rolling4w': rolling4w_results_df,
    'prophet': prophet_results_df,
    'sarima': sarima_results_df,
    'hw': hw_results_df,
    #'xgb': xgb_results_df,
    #'prophet+xgb': prophet_xgb_results_df,
    #'LSTM': lstm_results_df,
    #'MSTL': mstl_results_df,
    'neural_prophet': neural_prophet_results_df
}

all_results = []
for model_name, df in models.items():
    df['model'] = model_name
    all_results.append(df)

# Put all of the dataframes together into one dataframe for display
final_results_df = pd.concat(all_results, ignore_index=True)
# Make a pivot table so that we display rmse, mape and then each of the models and their results.
final_table = final_results_df.pivot(index='fold', columns='model', values=['rmse', 'mape'])
final_table.index = final_table.index.where(final_table.index != '-', 'mean')

final_table

rmse                                                             \
model   baseline         hw neural_prophet    prophet  rolling4w     sarima   
fold                                                                          
0      15.307003  17.167550      13.767870  12.928150  14.323744  15.174437   
1      11.914386  17.486979       9.628497   9.293687  12.719150  15.002202   
2      14.988228  17.312305       8.468593   7.988090  12.398157  12.875488   
3      20.818662  22.508395      10.982896   9.851386  19.240721  17.310608   
4      17.563835  19.906609      12.371418  10.863706  15.923646  14.457614   
5      23.244116  20.725266      11.030414  15.435334  17.378918  17.378153   
6      19.488233  22.480081      12.656647  11.555594  24.564601  14.693533   
7      21.222927  25.834222      13.416199  11.307567  20.989793  20.289146   
8      20.494910  23.345596      12.015590  10.504938  19.858428  20.150224   
9      21.193296  21.350550      12.227787   8.858023  18.798509  14.655524   
10     31.447761  23.476428      11.643435  15.301574  15.939618  18.099874   
11     22.102668  20.827749      10.225418   9.097185  13.952726  13.843936   
12     23.922867  18.618683      12.332529  11.121537  17.455044  16.878301   
13     19.713532  26.515049      11.528893  12.033741  19.822201  19.222472   
14     34.114645  26.458619      14.791254  18.991435  21.751026  21.417487   
15     26.451596  17.653833       7.994163   8.174644  13.761683  13.961373   
16     28.910678  15.215434      10.969603  11.818035  12.954591  10.138182   
17     35.881458  16.604841       9.196396  15.726014  19.309278  17.270045   
18     24.802787  13.591667      10.743974  11.783172   8.641676  10.547580   
19     18.947508  17.519523      10.274299  11.738618  15.194865  13.990687   
20     24.849791  11.074879      10.006320  10.670824  14.811916   6.956728   
21     22.731809  12.853506       6.249212   9.160873   8.280248   7.824108   
22     20.216302  13.647015       9.568202   9.815806  15.873383  11.674859   
23     30.933127  15.211508       6.523157  11.508407  11.649647  11.996826   
24     30.349771   6.909300       7.394597   9.012328   8.625129   5.305709   
25     26.045942  14.061492      10.787816  14.222667  13.177023  11.042151   
mean   23.371455  18.398349      10.645968  11.490898  15.669066  14.313740   

           mape                                                         
model  baseline        hw neural_prophet   prophet rolling4w    sarima  
fold                                                                    
0      0.285977  0.280495       0.229247  0.245960  0.239272  0.240537  
1      0.186347  0.255434       0.114966  0.142832  0.196755  0.201508  
2      0.250230  0.297496       0.142349  0.130246  0.196394  0.199047  
3      0.348457  0.356789       0.161305  0.143899  0.260374  0.267932  
4      0.256820  0.267117       0.159688  0.137638  0.220183  0.204142  
5      0.361006  0.310178       0.141677  0.215371  0.256428  0.257355  
6      0.266129  0.270543       0.157797  0.145927  0.247419  0.177519  
7      0.284166  0.311912       0.166526  0.135042  0.255484  0.245472  
8      0.301036  0.306865       0.163001  0.117917  0.253080  0.262065  
9      0.277230  0.274466       0.129494  0.110440  0.187194  0.176261  
10     0.498262  0.359001       0.166564  0.224857  0.229220  0.271150  
11     0.317831  0.281994       0.137190  0.116008  0.194994  0.187705  
12     0.359283  0.214809       0.113754  0.117579  0.200704  0.187011  
13     0.275826  0.313919       0.128031  0.120242  0.223940  0.212903  
14     0.766270  0.575918       0.268329  0.388540  0.478330  0.465972  
15     0.582483  0.338873       0.151492  0.153297  0.290997  0.295860  
16     0.723483  0.350441       0.214854  0.256895  0.308417  0.232142  
17     1.485229  0.653062       0.324309  0.558450  0.808842  0.716094  
18     0.767713  0.303153       0.229789  0.308906  0.170184  0.206923  
19     0.787489  0.500983       0.2

## Summary

In the above table, we see that the NeruralProphet / Prophet model had the best average RMSE. We also see that the next best were the SARIMA model and the Prophet + XGBoost models. Therefore, we will select the Prophet model for our modeling purposes and only maybe consider training a Prophet + XGBoost model to see if there is any improvement. We now know that the Prophet model performs the best, but we might ask if using an XGBoost model to predict residuals can lead to better model performance. In the above, we were unable to do that, but we were also using suboptimal parameters. 


Cautionary tales have been told about NeuralProphet and Prophet. For this reason, despite its performance here, we will be very careful to make sure that indeed these are the best models for the data available.

# Prophet Models with varying degrees of Initial Data

One might ask if there is too much noise at the start of the data. Afterall, 2020-2021 were still during the years where the effects of the COVID-19 pandemic were being felt. We consider what happens to Prophet's predictions as we drop data before 2020, 2021, 2022, 2023, and 2024. Of course, dropping data before 2020 just corresponds to the results above. We collect the RMSEs and MAPEs into a table and display the results in a markdown cell below at the end of this section. As a preview, we do not end up finding significant improvements in the model by dropping dates before a given year.

## Dropping <2020 data

In [109]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# This is should be 2251 - 366 since there was a leap year
print(len(rs)==2251-366)

date_range = pd.date_range(start="2020-01-01", end="2026-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

prophet20_results_df = pd.DataFrame(results)
prophet20_results_df.loc['mean'] = ['mean',  prophet20_results_df['rmse'].mean(), prophet20_results_df['mape'].mean()]
prophet20_results_df

False


23:25:20 - cmdstanpy - INFO - Chain [1] start processing
23:25:20 - cmdstanpy - INFO - Chain [1] done processing
23:25:21 - cmdstanpy - INFO - Chain [1] start processing
23:25:21 - cmdstanpy - INFO - Chain [1] done processing
23:25:22 - cmdstanpy - INFO - Chain [1] start processing
23:25:22 - cmdstanpy - INFO - Chain [1] done processing
23:25:23 - cmdstanpy - INFO - Chain [1] start processing
23:25:23 - cmdstanpy - INFO - Chain [1] done processing
23:25:24 - cmdstanpy - INFO - Chain [1] start processing
23:25:24 - cmdstanpy - INFO - Chain [1] done processing
23:25:25 - cmdstanpy - INFO - Chain [1] start processing
23:25:25 - cmdstanpy - INFO - Chain [1] done processing
23:25:26 - cmdstanpy - INFO - Chain [1] start processing
23:25:26 - cmdstanpy - INFO - Chain [1] done processing
23:25:27 - cmdstanpy - INFO - Chain [1] start processing
23:25:27 - cmdstanpy - INFO - Chain [1] done processing
23:25:28 - cmdstanpy - INFO - Chain [1] start processing
23:25:29 - cmdstanpy - INFO - Chain [1]

,fold,rmse,mape
0,0,12.928150,0.245960
1,1,9.293687,0.142832
2,2,7.988090,0.130246
3,3,9.851386,0.143899
4,4,10.863706,0.137638
5,5,15.435334,0.215371
6,6,11.555594,0.145927
7,7,11.307567,0.135042
8,8,10.504938,0.117917
9,9,8.858023,0.110440


## Dropping <2021 data

In [110]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2021-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# This is should be 2251 - 366 since there was a leap year
print(len(rs)==2251-366)

date_range = pd.date_range(start="2021-01-01", end="2026-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

prophet21_results_df = pd.DataFrame(results)
prophet21_results_df.loc['mean'] = ['mean',  prophet21_results_df['rmse'].mean(), prophet21_results_df['mape'].mean()]
prophet21_results_df

True


23:25:51 - cmdstanpy - INFO - Chain [1] start processing
23:25:52 - cmdstanpy - INFO - Chain [1] done processing
23:25:52 - cmdstanpy - INFO - Chain [1] start processing
23:25:52 - cmdstanpy - INFO - Chain [1] done processing
23:25:53 - cmdstanpy - INFO - Chain [1] start processing
23:25:53 - cmdstanpy - INFO - Chain [1] done processing
23:25:54 - cmdstanpy - INFO - Chain [1] start processing
23:25:54 - cmdstanpy - INFO - Chain [1] done processing
23:25:55 - cmdstanpy - INFO - Chain [1] start processing
23:25:55 - cmdstanpy - INFO - Chain [1] done processing
23:25:56 - cmdstanpy - INFO - Chain [1] start processing
23:25:56 - cmdstanpy - INFO - Chain [1] done processing
23:25:56 - cmdstanpy - INFO - Chain [1] start processing
23:25:57 - cmdstanpy - INFO - Chain [1] done processing
23:25:57 - cmdstanpy - INFO - Chain [1] start processing
23:25:58 - cmdstanpy - INFO - Chain [1] done processing
23:25:58 - cmdstanpy - INFO - Chain [1] start processing
23:25:58 - cmdstanpy - INFO - Chain [1]

,fold,rmse,mape
0,0,13.010372,0.242910
1,1,9.033825,0.137022
2,2,8.645587,0.138484
3,3,10.562346,0.155997
4,4,11.506908,0.145528
5,5,14.891346,0.199668
6,6,11.912402,0.158064
7,7,10.585563,0.127972
8,8,10.558701,0.110221
9,9,9.820695,0.125587


## Dropping <2022 data

In [111]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2022-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# This is should be 2251 - 366 - 365 since there was a leap year
print(len(rs)==2251-366-365)

date_range = pd.date_range(start="2022-01-01", end="2026-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

prophet22_results_df = pd.DataFrame(results)
prophet22_results_df.loc['mean'] = ['mean',  prophet22_results_df['rmse'].mean(), prophet22_results_df['mape'].mean()]
prophet22_results_df

True


23:26:19 - cmdstanpy - INFO - Chain [1] start processing
23:26:19 - cmdstanpy - INFO - Chain [1] done processing
23:26:19 - cmdstanpy - INFO - Chain [1] start processing
23:26:20 - cmdstanpy - INFO - Chain [1] done processing
23:26:20 - cmdstanpy - INFO - Chain [1] start processing
23:26:20 - cmdstanpy - INFO - Chain [1] done processing
23:26:21 - cmdstanpy - INFO - Chain [1] start processing
23:26:21 - cmdstanpy - INFO - Chain [1] done processing
23:26:22 - cmdstanpy - INFO - Chain [1] start processing
23:26:22 - cmdstanpy - INFO - Chain [1] done processing
23:26:23 - cmdstanpy - INFO - Chain [1] start processing
23:26:23 - cmdstanpy - INFO - Chain [1] done processing
23:26:23 - cmdstanpy - INFO - Chain [1] start processing
23:26:24 - cmdstanpy - INFO - Chain [1] done processing
23:26:24 - cmdstanpy - INFO - Chain [1] start processing
23:26:24 - cmdstanpy - INFO - Chain [1] done processing
23:26:25 - cmdstanpy - INFO - Chain [1] start processing
23:26:25 - cmdstanpy - INFO - Chain [1]

,fold,rmse,mape
0,0,13.133850,0.242441
1,1,10.281522,0.164575
2,2,8.522859,0.130986
3,3,11.265240,0.172244
4,4,11.851845,0.155127
5,5,14.860763,0.200533
6,6,12.302549,0.163929
7,7,11.167141,0.129801
8,8,10.589693,0.112496
9,9,9.622512,0.120706


## Dropping <2023 data

In [112]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2023-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# This is should be 2251 - 366 - 365 since there was a leap year
print(len(rs)==2251-366-365-365)

date_range = pd.date_range(start="2023-01-01", end="2026-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

prophet23_results_df = pd.DataFrame(results)
prophet23_results_df.loc['mean'] = ['mean',  prophet23_results_df['rmse'].mean(), prophet23_results_df['mape'].mean()]
prophet23_results_df

True


23:26:42 - cmdstanpy - INFO - Chain [1] start processing
23:26:42 - cmdstanpy - INFO - Chain [1] done processing
23:26:42 - cmdstanpy - INFO - Chain [1] start processing
23:26:42 - cmdstanpy - INFO - Chain [1] done processing
23:26:43 - cmdstanpy - INFO - Chain [1] start processing
23:26:43 - cmdstanpy - INFO - Chain [1] done processing
23:26:44 - cmdstanpy - INFO - Chain [1] start processing
23:26:44 - cmdstanpy - INFO - Chain [1] done processing
23:26:44 - cmdstanpy - INFO - Chain [1] start processing
23:26:44 - cmdstanpy - INFO - Chain [1] done processing
23:26:45 - cmdstanpy - INFO - Chain [1] start processing
23:26:45 - cmdstanpy - INFO - Chain [1] done processing
23:26:46 - cmdstanpy - INFO - Chain [1] start processing
23:26:46 - cmdstanpy - INFO - Chain [1] done processing
23:26:46 - cmdstanpy - INFO - Chain [1] start processing
23:26:46 - cmdstanpy - INFO - Chain [1] done processing
23:26:47 - cmdstanpy - INFO - Chain [1] start processing
23:26:47 - cmdstanpy - INFO - Chain [1]

,fold,rmse,mape
0,0,15.709343,0.247035
1,1,16.025910,0.244400
2,2,9.128428,0.137109
3,3,11.076445,0.166157
4,4,11.131715,0.143922
5,5,12.350706,0.153552
6,6,13.672877,0.160912
7,7,13.609114,0.135461
8,8,10.864818,0.128133
9,9,8.163798,0.103427


## Dropping <2024 data

In [113]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2024-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# This is should be 2251 - 366 - 365 since there was a leap year
print(len(rs)==2251-366-365-365-366)

date_range = pd.date_range(start="2024-01-01", end="2026-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

prophet24_results_df = pd.DataFrame(results)
prophet24_results_df.loc['mean'] = ['mean',  prophet24_results_df['rmse'].mean(), prophet24_results_df['mape'].mean()]
prophet24_results_df

False


23:27:01 - cmdstanpy - INFO - Chain [1] start processing
23:27:01 - cmdstanpy - INFO - Chain [1] done processing
23:27:01 - cmdstanpy - INFO - Chain [1] start processing
23:27:01 - cmdstanpy - INFO - Chain [1] done processing
23:27:02 - cmdstanpy - INFO - Chain [1] start processing
23:27:02 - cmdstanpy - INFO - Chain [1] done processing
23:27:02 - cmdstanpy - INFO - Chain [1] start processing
23:27:02 - cmdstanpy - INFO - Chain [1] done processing
23:27:03 - cmdstanpy - INFO - Chain [1] start processing
23:27:03 - cmdstanpy - INFO - Chain [1] done processing
23:27:03 - cmdstanpy - INFO - Chain [1] start processing
23:27:03 - cmdstanpy - INFO - Chain [1] done processing
23:27:04 - cmdstanpy - INFO - Chain [1] start processing
23:27:04 - cmdstanpy - INFO - Chain [1] done processing
23:27:04 - cmdstanpy - INFO - Chain [1] start processing
23:27:04 - cmdstanpy - INFO - Chain [1] done processing
23:27:05 - cmdstanpy - INFO - Chain [1] start processing
23:27:05 - cmdstanpy - INFO - Chain [1]

,fold,rmse,mape
0,0,25.523107,0.373311
1,1,18.606959,0.281424
2,2,9.190205,0.122313
3,3,11.587698,0.152301
4,4,13.146713,0.154916
5,5,11.645020,0.119177
6,6,13.553334,0.155016
7,7,12.622754,0.132684
8,8,10.243038,0.110119
9,9,10.491212,0.123024


## Results Table

In [114]:
# We make a dictionary of models and their results to make it easier to iterate over. 
# Make sure to add to this if writing new models in.
models = {
    'prophet24': prophet24_results_df,
    'prophet23': prophet23_results_df,
    'prophet22': prophet22_results_df,
    'prophet21': prophet21_results_df,
    'prophet20': prophet20_results_df,
}

all_results = []
for model_name, df in models.items():
    df['model'] = model_name
    all_results.append(df)

# Put all of the dataframes together into one dataframe for display
final_results_df = pd.concat(all_results, ignore_index=True)
# Make a pivot table so that we display rmse, mape and then each of the models and their results.
final_table = final_results_df.pivot(index='fold', columns='model', values=['rmse', 'mape'])
final_table.index = final_table.index.where(final_table.index != '-', 'mean')

final_table

rmse                                                  mape  \
model  prophet20  prophet21  prophet22  prophet23  prophet24 prophet20   
fold                                                                     
0      12.928150  13.010372  13.133850  15.709343  25.523107  0.245960   
1       9.293687   9.033825  10.281522  16.025910  18.606959  0.142832   
2       7.988090   8.645587   8.522859   9.128428   9.190205  0.130246   
3       9.851386  10.562346  11.265240  11.076445  11.587698  0.143899   
4      10.863706  11.506908  11.851845  11.131715  13.146713  0.137638   
5      15.435334  14.891346  14.860763  12.350706  11.645020  0.215371   
6      11.555594  11.912402  12.302549  13.672877  13.553334  0.145927   
7      11.307567  10.585563  11.167141  13.609114  12.622754  0.135042   
8      10.504938  10.558701  10.589693  10.864818  10.243038  0.117917   
9       8.858023   9.820695   9.622512   8.163798  10.491212  0.110440   
10     15.301574  17.628646  16.129086  16.137087  17.328348  0.224857   
11      9.097185   9.238740   9.369240  10.450350  13.903741  0.116008   
12     11.121537  10.967539  11.144263  11.930778  15.690875  0.117579   
13     12.033741  12.185596  11.676091  11.320118  12.322772  0.120242   
14     18.991435  19.020887  18.177308  20.356476  32.289429  0.388540   
15      8.174644   7.632799   6.496359   6.987397  21.138268  0.153297   
16     11.818035  11.143183   8.852227  10.384570  22.706691  0.256895   
17     15.726014  15.384071  10.592132   9.904990  25.815500  0.558450   
18     11.783172  12.222960  11.604328  10.990474  13.930732  0.308906   
19     11.738618  13.478664  13.618805  13.839817   9.798919  0.360274   
20     10.670824  11.215044  11.152246  10.594978  14.077004  0.409166   
21      9.160873  10.054518  10.042451   9.624049  10.529644  0.409142   
22      9.815806  11.225782  10.927968  10.466579  14.910636  0.241310   
23     11.508407  10.335301  11.389266  12.792121  12.932985  0.505464   
24      9.012328   8.914322   9.829001  10.238909   8.782563  0.362696   
25     14.222667  14.771585  14.235184  13.973552  14.830314  0.495670   
mean   11.490898  11.767207  11.493613  11.989438  15.292249  0.252068   

                                               
model prophet21 prophet22 prophet23 prophet24  
fold                                           
0      0.242910  0.242441  0.247035  0.373311  
1      0.137022  0.164575  0.244400  0.281424  
2      0.138484  0.130986  0.137109  0.122313  
3      0.155997  0.172244  0.166157  0.152301  
4      0.145528  0.155127  0.143922  0.154916  
5      0.199668  0.200533  0.153552  0.119177  
6      0.158064  0.163929  0.160912  0.155016  
7      0.127972  0.129801  0.135461  0.132684  
8      0.110221  0.112496  0.128133  0.110119  
9      0.125587  0.120706  0.103427  0.123024  
10     0.263745  0.238550  0.238568  0.256492  
11     0.113755  0.115348  0.137400  0.192068  
12     0.108370  0.127574  0.150389  0.224922  
13     0.119474  0.120870  0.120258  0.162417  
14     0.388210  0.370159  0.419509  0.694599  
15     0.135494  0.107656  0.126587  0.455334  
16     0.245061  0.179165  0.209112  0.520799  
17     0.535264  0.357378  0.340599  0.972523  
18     0.324015  0.297048  0.270380  0.355581  
19     0.419117  0.422809  0.426582  0.319298  
20     0.448994  0.452295  0.426254  0.564992  
21     0.472847  0.491401  0.429285  0.461792  
22     0.275848  0.271445  0.253736  0.354487  
23     0.397077  0.481136  0.589772  0.604895  
24     0.357543  0.394995  0.389187  0.342124  
25     0.525176  0.497270  0.493095  0.530622  
mean   0.256594  0.250690  0.255416  0.336047

The results above above indicate that the Prophet model *does not* on average improve significantly as we start dropping data before a given year. We can see this in some folds e.g. fold 0, 1, 2. We also see that there are some folds where the RMSE basically stays the same. Then there are folds like folds 15 and 11 where things seem to improve by dropping some years. The best average RMSE, however, does occur if we drop all pre-2022 data though the improvement is very minor. One might hypothesize that this was due to the lfiting of COVID-19 lockdowns.

# Comparing Prophet Model versus an XGBoosted Prophet Model

This section of code can be long so we summarize. First, we saw from before that Prophet performed very well. We also saw that Prophet + XGBoost on residuals performed slightly worse. This could be due to not tuning the parameters very well. So, we might instead try to see if we can get better predictions by actually doing some feature engineering and hyperparameter tuning of XGBoost.

There are a few things we must be careful about.

1. Remember that we are evaluating based on walk-forward validation. That means we fix a training set, and then evaluate 14 days out. Then for the next fold, we add those 14 days to our training set and evaluate 14 days out again. Then add those 14 days to the training set and evaluate 14 days out again. Then we repeat until we have done 26 folds.

2. Suppose we fix a fold so we have a training set and a test set decided. We first fit Prophet to the training set. Then determine the residuals there. Using the residuals, we then train an XGBoost model fitted to the residuals. Then, we use both the Prophet model and the XGBoost model to forecast 14 days out and we evaluate against the test set.

3. We want **clear** evidence of improvement. So, we had proceeded as before and used a TimeSeriesSplit. In training the XGBoost model, we need to be careful of overfitting. We use Optuna to tune the hyperparameters and manually select features until we see improvement. The benefit of using Optuna is that we have written our objective function to avoid overfitting to the residuals' training set. 

4. Even if we have *clear* evidence of improvement, we also must ensure that the added **complexity** is worth it. Training an XGBoost model is quite time intensive and having to tune hyperparameters for each fold is already a lot of work. The value of Optuna here is that we can limit the number of trials.

## Reimport the Data

We reimport the data so that we can focus only on running this section's code as opposed to having to rerun from the start.

In [115]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

In [116]:
rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])

In [117]:
# Start by cutting off data before 2020-01-01 and after 2026-02-28.
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']

In [118]:
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')

In [119]:
## This is 2251 which equals the number of days from 2020-01-01 to 2026-02-28
len(rs)

2251

In [120]:
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

In [121]:
rs

,ds,y
0,2020-01-01,17
1,2020-01-02,40
2,2020-01-03,41
3,2020-01-04,25
4,2020-01-05,17
...,...,...
2246,2026-02-24,26
2247,2026-02-25,30
2248,2026-02-26,40
2249,2026-02-27,38


## Repeat the Computations for Prophet

In [122]:
# Create a date range covering 2020 through end of 2025
date_range = pd.date_range(start="2020-01-01", end="2026-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

# Build the DataFrame in the same structure as your original
federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1,
})

holidays = federal_holidays

# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Calculate MAPE
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# Convert results to a datafrane
prophet_results_df = pd.DataFrame(results)

prophet_results_df.loc['mean'] = ['mean',  prophet_results_df['rmse'].mean(), prophet_results_df['mape'].mean()]

23:27:17 - cmdstanpy - INFO - Chain [1] start processing
23:27:17 - cmdstanpy - INFO - Chain [1] done processing
23:27:18 - cmdstanpy - INFO - Chain [1] start processing
23:27:18 - cmdstanpy - INFO - Chain [1] done processing
23:27:19 - cmdstanpy - INFO - Chain [1] start processing
23:27:19 - cmdstanpy - INFO - Chain [1] done processing
23:27:20 - cmdstanpy - INFO - Chain [1] start processing
23:27:20 - cmdstanpy - INFO - Chain [1] done processing
23:27:21 - cmdstanpy - INFO - Chain [1] start processing
23:27:21 - cmdstanpy - INFO - Chain [1] done processing
23:27:22 - cmdstanpy - INFO - Chain [1] start processing
23:27:23 - cmdstanpy - INFO - Chain [1] done processing
23:27:23 - cmdstanpy - INFO - Chain [1] start processing
23:27:24 - cmdstanpy - INFO - Chain [1] done processing
23:27:24 - cmdstanpy - INFO - Chain [1] start processing
23:27:25 - cmdstanpy - INFO - Chain [1] done processing
23:27:25 - cmdstanpy - INFO - Chain [1] start processing
23:27:26 - cmdstanpy - INFO - Chain [1]

## Adding Features to XGBoost

In [123]:
def create_features(df):
    """
    Create time series features based on time series index.
    """
    df = df.copy()
    df['dayofweek'] = df.index.dayofweek
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['dayofyear'] = df.index.dayofyear
    df['dayofmonth'] = df.index.day
    df['weekofyear'] = df.index.isocalendar().week
    return df

def add_cyclic(df):
    target_map = df['y'].to_dict()
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek']/7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek']/7)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
    return df

def add_lags(df):
    target_map = df['y'].to_dict()
    df['lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
    df['lag2'] = (df.index - pd.Timedelta('2 days')).map(target_map)
    df['lag3'] = (df.index - pd.Timedelta('3 days')).map(target_map)
    df['lag4'] = (df.index - pd.Timedelta('4 days')).map(target_map)
    df['lag5'] = (df.index - pd.Timedelta('5 days')).map(target_map)
    df['lag6'] = (df.index - pd.Timedelta('6 days')).map(target_map)
    df['lag7'] = (df.index - pd.Timedelta('7 days')).map(target_map)
    df['lag8'] = (df.index - pd.Timedelta('8 days')).map(target_map)
    df['lag9'] = (df.index - pd.Timedelta('9 days')).map(target_map)
    df['lag10'] = (df.index - pd.Timedelta('10 days')).map(target_map)
    df['lag11'] = (df.index - pd.Timedelta('11 days')).map(target_map)
    df['lag12'] = (df.index - pd.Timedelta('12 days')).map(target_map)
    df['lag13'] = (df.index - pd.Timedelta('13 days')).map(target_map)
    df['lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    return df

def add_seasonal_lags(df):
    target_map = df['y'].to_dict()
    df['lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)

    df['lag362'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag363'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag364'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag366'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag367'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    
    df['lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)
    df['lag1095'] = (df.index - pd.Timedelta('1095 days')).map(target_map)
    df['lag1460'] = (df.index - pd.Timedelta('1460 days')).map(target_map)
    df['lag1825'] = (df.index - pd.Timedelta('1825 days')).map(target_map)
    return df

def add_moving_averages(df):
    df = df.copy()
    
    # Ensure sorted by date
    df = df.sort_index()
    
    # Moving averages (using previous values only)
    df['ma7'] = df['y'].shift(1).rolling(window=7).mean()
    df['ma30'] = df['y'].shift(1).rolling(window=30).mean()
    df['ma60'] = df['y'].shift(1).rolling(window=60).mean()
    df['ma90'] = df['y'].shift(1).rolling(window=90).mean()
    df['ma120'] = df['y'].shift(1).rolling(window=120).mean()
    df['ma150'] = df['y'].shift(1).rolling(window=150).mean()
    df['ma180'] = df['y'].shift(1).rolling(window=180).mean()
    df['ma365'] = df['y'].shift(1).rolling(window=365).mean()
    
    return df


In the next two code block, we add weather data to the data set. This is not optimized i.e. we just obtain the weather data in Manhattan and hope that it is representative of the average weather over the whole city.

In [124]:
## Add weather data.

import requests
import pandas as pd

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd

else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [125]:
def add_weather_data(df, wd):
    df = df.copy()
    wd = wd.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    wd.index = pd.to_datetime(wd.index)
    
    # Drop unnecessary column
    if "time" in wd.columns:
        wd = wd.drop(columns=["time"])
    
    # Join on date index
    df = df.join(wd, how="left")
    
    return df

In [126]:
from pandas.tseries.holiday import USFederalHolidayCalendar

def add_federal_holidays(df, custom_holidays=None):
    df = df.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df.index.min(), end=df.index.max())
    
    if custom_holidays:
        for d in custom_holidays:
            if len(d) == 5:  # MM-DD format → recurring annually
                years = df.index.year.unique()
                for y in years:
                    holidays = holidays.append(pd.to_datetime([f"{y}-{d}"]))
            else:  # YYYY-MM-DD → one specific date
                holidays = holidays.append(pd.to_datetime([d]))
    
    holidays = holidays.drop_duplicates().sort_values()
    
    df["is_federal_holiday"] = df.index.isin(holidays).astype(int)
    
    return df

In [127]:
def add_law_flag(df, law_name: str, start_date: str):
    # Adds a binary column to indicate when a new law is active.
    df = df.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    
    # Convert start_date to datetime
    start_dt = pd.to_datetime(start_date)
    
    # Create binary column: 1 if date >= start_date, else 0
    df[law_name] = (df.index >= start_dt).astype(int)
    
    return df

In [128]:
# This must be run importing weather data

def add_more_weather_feature(df):
    target_map = df['apparent_temperature_min'].to_dict()
    df['apparent_temperature_min_lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
    df['apparent_temperature_min_lag2'] = (df.index - pd.Timedelta('2 days')).map(target_map)
    df['apparent_temperature_min_lag7'] = (df.index - pd.Timedelta('7 days')).map(target_map)
    
    df['apparent_temperature_min_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['apparent_temperature_min_lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df['apparent_temperature_min_lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df['apparent_temperature_min_lag17'] = (df.index - pd.Timedelta('17 days')).map(target_map)

    df['apparent_temperature_min_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['apparent_temperature_min_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['apparent_temperature_min_lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['apparent_temperature_min_lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['apparent_temperature_min_lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['apparent_temperature_min_lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)
    df['apparent_temperature_min_lag210'] = (df.index - pd.Timedelta('210 days')).map(target_map)
    df['apparent_temperature_min_lag240'] = (df.index - pd.Timedelta('240 days')).map(target_map)
    df['apparent_temperature_min_lag270'] = (df.index - pd.Timedelta('270 days')).map(target_map)
    df['apparent_temperature_min_lag300'] = (df.index - pd.Timedelta('300 days')).map(target_map)
    df['apparent_temperature_min_lag330'] = (df.index - pd.Timedelta('330 days')).map(target_map)
    df['apparent_temperature_min_lag360'] = (df.index - pd.Timedelta('360 days')).map(target_map)
    df['apparent_temperature_min_lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['apparent_temperature_min_lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)

    target_map = df['temperature_2m_max'].to_dict()
    df['temperature_2m_max_lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
    df['temperature_2m_max_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['temperature_2m_max_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['temperature_2m_max_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)

    return df

In [129]:
def add_forecast_features(df,forecast):
    for column in forecast.columns:
        df[column] = forecast[column]

    return df

In [130]:
forecast

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,Christmas Day,Christmas Day_lower,Christmas Day_upper,Christmas Day (observed),...,weekly,weekly_lower,weekly_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2020-01-01,39.222128,-6.269393,27.842449,39.222128,39.222128,0.0,0.0,0.0,0.0,...,7.398973,7.398973,7.398973,-20.580561,-20.580561,-20.580561,0.0,0.0,0.0,10.645006
1,2020-01-02,39.247510,6.810800,41.443600,39.247510,39.247510,0.0,0.0,0.0,0.0,...,6.542359,6.542359,6.542359,-19.851061,-19.851061,-19.851061,0.0,0.0,0.0,24.816030
2,2020-01-03,39.272891,2.257890,36.963269,39.272891,39.272891,0.0,0.0,0.0,0.0,...,-0.863579,-0.863579,-0.863579,-19.104852,-19.104852,-19.104852,0.0,0.0,0.0,19.304460
3,2020-01-04,39.298272,-14.168405,19.750633,39.298272,39.298272,0.0,0.0,0.0,0.0,...,-18.105707,-18.105707,-18.105707,-18.351982,-18.351982,-18.351982,0.0,0.0,0.0,2.840583
4,2020-01-05,39.323654,-11.271849,22.167621,39.323654,39.323654,0.0,0.0,0.0,0.0,...,-16.166179,-16.166179,-16.166179,-17.602804,-17.602804,-17.602804,0.0,0.0,0.0,5.554671
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2246,2026-02-24,44.726814,26.986866,60.296482,44.726814,44.726814,0.0,0.0,0.0,0.0,...,9.270101,9.270101,9.270101,-9.449222,-9.449222,-9.449222,0.0,0.0,0.0,44.547694
2247,2026-02-25,44.682406,25.335714,59.433748,44.682406,44.682406,0.0,0.0,0.0,0.0,...,7.398973,7.398973,7.398973,-8.906990,-8.906990,-8.906990,0.0,0.0,0.0,43.174389
2248,2026-02-26,44.637997,26.316335,60.978669,44.637997,44.637997,0.0,0.0,0.0,0.0,...,6.542359,6.542359,6.542359,-8.405419,-8.405419,-8.405419,0.0,0.0,0.0,42.774938
2249,2026-02-27,44.593589,18.718669,53.444950,44.593589,44.593589,0.0,0.0,0.0,0.0,...,-0.863579,-0.863579,-0.863579,-7.948633,-7.948633,-7.948633,0.0,0.0,0.0,35.781376


## Features for XGBoost

In [131]:
FIRST_FEATURES = ['apparent_temperature_min_lag1',
            'temperature_2m_max_lag1',
            'dayofyear', 
            'temperature_2m_max_lag60'
            ]

## Add default parameters for XGBoost

In [132]:
params = {'objective': 'reg:squarederror',
         'eval_metric': 'rmse',
         'booster': 'gbtree',
         'base_score': -5, 
         'n_estimators': 3000, 
         'min_child_weight': 5, 
         'learning_rate': 0.001,
         'max_depth': 7,
         #'subsample': 0.9,
         #'colsample_bytree': 0.7,
         #'colsample_bylevel': 0.6, 
         #'colsample_bynode': 0.9, 
         #'reg_alpha': 4, 
         #'gamma': 1.5, 
         #'reg_lambda': 4.5,
         #'early_stopping_rounds': 100, 
        }

In [133]:
## Old code to evaluate XGBoost + Prophet model.

# results = []

# for i, (train_index, test_index) in enumerate(tscv.split(rs)):
#     # Split the dataset into training and testing sets
#     train = rs.iloc[train_index]
#     test = rs.iloc[test_index]
    
#     # Fit Prophet on the training data
#     model = Prophet(holidays=holidays)
#     model.add_country_holidays(country_name='US')
#     model.fit(train)
    
#     # Make predictions on the training set to calculate residuals
#     train_future = model.make_future_dataframe(periods=0, freq='D')  # Use periods=0 to only use the training data
#     train_forecast = model.predict(train_future)
    
#     # Calculate residuals (actual - predicted) on the training data
#     train_residuals = train['y'].values - train_forecast['yhat'][:len(train)].values
    
#     # Build a new DataFrame of residuals
#     residuals_df = pd.DataFrame({'ds': train['ds'], 'y': train_residuals })

#     train = train.set_index('ds')
#     train.index = pd.to_datetime(train.index)
#     train = create_features(train)
#     train = add_cyclic(train)
#     train = add_lags(train)
#     train = add_seasonal_lags(train)
#     train = add_moving_averages(train)
#     train = add_weather_data(train,wd)
#     train = add_more_weather_feature(train)
#     train = add_federal_holidays(train, custom_holidays = ['12-31'])
#     train = add_law_flag(train, law_name='Trash_Law', start_date = '2024-03-01')
#     train = add_law_flag(train, law_name = 'New_Trash_Law', start_date = '2024-11-01')
#     train = add_law_flag(train, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
#     train = add_law_flag(train, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

#     X_train_residuals = train[FEATURES]
#     y_train_residuals = residuals_df['y']

    
    
#     xgb_model = xgb.XGBRegressor(**params)
#     xgb_model.fit(X_train_residuals, y_train_residuals)
    

#     test = test.set_index('ds')
#     test.index = pd.to_datetime(test.index)
#     test = create_features(test)
#     test = add_cyclic(test)
#     test = add_lags(test)
#     test = add_seasonal_lags(test)
#     test = add_moving_averages(test)
#     test = add_weather_data(test,wd)
#     test = add_more_weather_feature(test)
#     test = add_federal_holidays(test, custom_holidays = ['12-31'])
#     test = add_law_flag(test, law_name='Trash_Law', start_date = '2024-03-01')
#     test = add_law_flag(test, law_name = 'New_Trash_Law', start_date = '2024-11-01')
#     test = add_law_flag(test, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
#     test = add_law_flag(test, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

#     # Predict residuals using XGBoost for the test set
#     X_test = test[FEATURES]  # Features for the test set
#     xgb_residual_preds = xgb_model.predict(X_test)
    
#     # Forecast using Prophet on the test set
#     future = model.make_future_dataframe(periods=len(test), freq='D')
#     prophet_forecast = model.predict(future)
    
#     # Combine Prophet's forecast and XGBoost's residual prediction
#     y_pred = prophet_forecast['yhat'][-len(test):].values + xgb_residual_preds
    
#     # Actual values for the test set
#     y_true = test['y'].values
    
#     # Calculate RMSE
#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
#     # Calculate MAPE
#     mape = mean_absolute_percentage_error(y_true, y_pred)
    
#     # Store the results for this fold
#     results.append({'fold': i, 'rmse': rmse, 'mape': mape})
    
    
#     ## Uncomment code below if you want to have plots on feature importance.
#     # fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(10, 30))
#     # plot_importance(xgb_model, ax=ax1, importance_type='gain')
#     # ax1.set_title('Gain-based Importance', fontsize=12)

#     # plot_importance(xgb_model, ax=ax2, importance_type='weight')
#     # ax2.set_title('Split-based Importance', fontsize=12)

#     # plot_importance(xgb_model, ax=ax3, importance_type='cover')
#     # ax3.set_title('Cover Importance', fontsize=12)

#     # plot_importance(xgb_model, ax=ax4, importance_type='total_gain')
#     # ax4.set_title('Total Gain Importance', fontsize=12)

#     # plot_importance(xgb_model, ax=ax5, importance_type='total_cover')
#     # ax5.set_title('Total Cover Importance', fontsize=12)

    
#     plt.show()

# # Convert the results into a DataFrame
# prophet_xgb_results_df = pd.DataFrame(results)

# prophet_xgb_results_df = pd.DataFrame(results)
# mean_rmse = prophet_xgb_results_df['rmse'].mean()
# mean_mape = prophet_xgb_results_df['mape'].mean()
# prophet_xgb_results_df.loc['mean'] = ['mean', mean_rmse, mean_mape]

## Optuna tuning before each fold for best hyperparameters

The following is a very long code block and probably takes quite a long time to run so I've tried my best to break down the workings of the code with tons of comments.

In [134]:
import optuna
from optuna.exceptions import TrialPruned
from sklearn.model_selection import train_test_split

results = []

# def objective(trial, X_train_residuals, y_train_residuals):
#     param = {'objective': 'reg:squarederror',
#          'eval_metric': 'rmse',
#          'booster': 'gbtree',
#         'n_estimators': trial.suggest_int('n_estimators', 700, 1000),
#         'max_depth': trial.suggest_int('max_depth', 1, 5),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 2, log=True),
#         'min_child_weight': trial.suggest_int('min_child_weight', 4, 10),
#         'subsample': trial.suggest_float('subsample', 0.5, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
#         'gamma': trial.suggest_float('gamma', 0, 5),
#         'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
#         'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
#         'random_state': 42
#     }

#     tscv_inner = TimeSeriesSplit(n_splits=5)  # small CV on fold

#     rmses = []
#     for tr_idx, val_idx in tscv_inner.split(X_train_residuals):
#         X_tr, X_val = X_train_residuals.iloc[tr_idx], X_train_residuals.iloc[val_idx]
#         y_tr, y_val = y_train_residuals.iloc[tr_idx], y_train_residuals.iloc[val_idx]

#         model = xgb.XGBRegressor(**param)
#         model.fit(X_tr, y_tr)
#         y_pred = model.predict(X_val)
#         rmses.append(np.sqrt(mean_squared_error(y_val, y_pred)))

#     return np.mean(rmses)

def objective(trial, X_train_residuals, y_train_residuals):
    param = {'objective': 'reg:squarederror',
             'eval_metric': 'rmse',
             'booster': 'gbtree',
             'n_estimators': trial.suggest_int('n_estimators', 700, 1000),
             'max_depth': trial.suggest_int('max_depth', 1, 5),
             'learning_rate': trial.suggest_float('learning_rate', 0.01, 2, log=True),
             'min_child_weight': trial.suggest_int('min_child_weight', 4, 10),
             'subsample': trial.suggest_float('subsample', 0.5, 1.0),
             'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
             'gamma': trial.suggest_float('gamma', 0, 5),
             'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
             'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
             'random_state': 42
    }

    tscv_inner = tscv
    rmses = []

    for fold, (tr_idx, val_idx) in enumerate(tscv_inner.split(X_train_residuals), 1):
        X_tr, X_val = X_train_residuals.iloc[tr_idx], X_train_residuals.iloc[val_idx]
        y_tr, y_val = y_train_residuals.iloc[tr_idx], y_train_residuals.iloc[val_idx]

        model = xgb.XGBRegressor(**param)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        y_pred = model.predict(X_val)
        fold_rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        rmses.append(fold_rmse)

        trial.report(np.mean(rmses), fold)
        if trial.should_prune():
            raise TrialPruned(f"Trial was pruned at fold {fold}.")

    return np.mean(rmses)

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    # Split the dataset
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    # Fit Prophet
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')
    model.fit(train)
    
    # Calculate residuals
    train_future = model.make_future_dataframe(periods=0, freq='D')
    train_forecast = model.predict(train_future)
    train_residuals = train['y'].values - train_forecast['yhat'][:len(train)].values
    residuals_df = pd.DataFrame({'ds': train['ds'], 'y': train_residuals})
    
    # Feature engineering
    train = train.set_index('ds')
    train.index = pd.to_datetime(train.index)
    train = create_features(train)
    train = add_cyclic(train)
    train = add_lags(train)
    train = add_seasonal_lags(train)
    train = add_moving_averages(train)
    train = add_weather_data(train, wd)
    train = add_more_weather_feature(train)
    train = add_federal_holidays(train, custom_holidays=['12-31'])
    train = add_law_flag(train, law_name='Trash_Law', start_date='2024-03-01')
    train = add_law_flag(train, law_name='New_Trash_Law', start_date='2024-11-01')
    train = add_law_flag(train, law_name='Rat_Mitigation_Zone', start_date='2023-07-07')
    train = add_law_flag(train, law_name='Rat_Czar_Appointed', start_date='2023-04-12')
    train = add_forecast_features(train, train_forecast)

    FEATURES = FIRST_FEATURES + list(train_forecast.columns)
    FEATURES = [x for x in FEATURES if x not in ['ds', 'Christmas Day',
    'Christmas Day_lower', 'Christmas Day_upper', 'Christmas Day (observed)',
    'Christmas Day (observed)_lower', 'Christmas Day (observed)_upper',
    'Columbus Day', 'Columbus Day_lower', 'Columbus Day_upper',
    'Independence Day', 'Independence Day_lower', 'Independence Day_upper', 'Independence Day (observed)','Independence Day (observed)_lower', 'Independence Day (observed)_upper',
    'Juneteenth National Independence Day', 'Juneteenth National Independence Day_lower', 'Juneteenth National Independence Day_upper', 
    'Juneteenth National Independence Day (observed)', 'Juneteenth National Independence Day (observed)_lower', 'Juneteenth National Independence Day (observed)_upper',
    'Labor Day', 'Labor Day_lower', 'Labor Day_upper',
    'Martin Luther King Jr. Day', 'Martin Luther King Jr. Day_lower', 'Martin Luther King Jr. Day_upper',
    'Memorial Day', 'Memorial Day_lower', 'Memorial Day_upper',
    "New Year's Day", "New Year's Day_lower", "New Year's Day_upper",
    "New Year's Day (observed)", "New Year's Day (observed)_lower", "New Year's Day (observed)_upper",
    'Thanksgiving Day', 'Thanksgiving Day_lower', 'Thanksgiving Day_upper',
    'Veterans Day', 'Veterans Day_lower', 'Veterans Day_upper',
    'Veterans Day (observed)', 'Veterans Day (observed)_lower', 'Veterans Day (observed)_upper',
    "Washington's Birthday", "Washington's Birthday_lower", "Washington's Birthday_upper",
    'federal_us', 'federal_us_lower', 'federal_us_upper',
    'holidays', 'holidays_lower', 'holidays_upper',
    'multiplicative_terms', 'multiplicative_terms_lower', 'multiplicative_terms_upper']]

    X_train_residuals = train[FEATURES]
    y_train_residuals = residuals_df['y']

    best_params = params
    # Uncomment the next 4 lines to include Optuna hyperparameter tuning. We used Optuna because our parameter space is quite large.
    # study = optuna.create_study(direction='minimize')
    # study.optimize(lambda trial: objective(trial, X_train_residuals, y_train_residuals), n_trials=10)
    # best_params = study.best_params 
    # best_params['random_state'] = 42

    # Train XGBoost with best parameters from Optuna.
    xgb_model = xgb.XGBRegressor(**best_params)
    xgb_model.fit(X_train_residuals, y_train_residuals)
    
    # Add the features we will use for XGBoost. Again, we are using XGBoost to try and predict the residuals of the Prophet model.
    test = test.set_index('ds')
    test.index = pd.to_datetime(test.index)
    test = create_features(test)
    test = add_cyclic(test)
    test = add_lags(test)
    test = add_seasonal_lags(test)
    test = add_moving_averages(test)
    test = add_weather_data(test, wd)
    test = add_more_weather_feature(test)
    test = add_federal_holidays(test, custom_holidays=['12-31'])
    test = add_law_flag(test, law_name='Trash_Law', start_date='2024-03-01')
    test = add_law_flag(test, law_name='New_Trash_Law', start_date='2024-11-01')
    test = add_law_flag(test, law_name='Rat_Mitigation_Zone', start_date='2023-07-07')
    test = add_law_flag(test, law_name='Rat_Czar_Appointed', start_date='2023-04-12')


    
    # Have prophet make a forecast on the test set which is 14 days.
    future = model.make_future_dataframe(periods=len(test), freq='D')
    prophet_forecast = model.predict(future)
    test = add_forecast_features(test, prophet_forecast)

    # Predict the residuals of Prophet for test set
    X_test = test[FEATURES]
    xgb_residual_preds = xgb_model.predict(X_test)
    
    # Add the forecast of the residuals by XGBoost for 14 days.
    y_pred = prophet_forecast['yhat'][-len(test):].values + xgb_residual_preds
    y_true = test['y'].values

    # Obtain metrics for our forecast against the actuals.
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# Results DataFrame
prophet_xgb_results_df = pd.DataFrame(results)
mean_rmse = prophet_xgb_results_df['rmse'].mean()
mean_mape = prophet_xgb_results_df['mape'].mean()
prophet_xgb_results_df.loc['mean'] = ['mean', mean_rmse, mean_mape]

23:27:49 - cmdstanpy - INFO - Chain [1] start processing
23:27:49 - cmdstanpy - INFO - Chain [1] done processing
23:27:55 - cmdstanpy - INFO - Chain [1] start processing
23:27:56 - cmdstanpy - INFO - Chain [1] done processing
23:28:02 - cmdstanpy - INFO - Chain [1] start processing
23:28:02 - cmdstanpy - INFO - Chain [1] done processing
23:28:09 - cmdstanpy - INFO - Chain [1] start processing
23:28:09 - cmdstanpy - INFO - Chain [1] done processing
23:28:16 - cmdstanpy - INFO - Chain [1] start processing
23:28:16 - cmdstanpy - INFO - Chain [1] done processing
23:28:23 - cmdstanpy - INFO - Chain [1] start processing
23:28:23 - cmdstanpy - INFO - Chain [1] done processing
23:28:29 - cmdstanpy - INFO - Chain [1] start processing
23:28:29 - cmdstanpy - INFO - Chain [1] done processing
23:28:35 - cmdstanpy - INFO - Chain [1] start processing
23:28:36 - cmdstanpy - INFO - Chain [1] done processing
23:28:42 - cmdstanpy - INFO - Chain [1] start processing
23:28:42 - cmdstanpy - INFO - Chain [1]

In [135]:
# here were the features we were using. tune them!
FEATURES

['apparent_temperature_min_lag1',
 'temperature_2m_max_lag1',
 'dayofyear',
 'temperature_2m_max_lag60',
 'trend',
 'yhat_lower',
 'yhat_upper',
 'trend_lower',
 'trend_upper',
 'additive_terms',
 'additive_terms_lower',
 'additive_terms_upper',
 'weekly',
 'weekly_lower',
 'weekly_upper',
 'yearly',
 'yearly_lower',
 'yearly_upper',
 'yhat']

In [136]:
# X_train_residuals.plot(figsize=(24, 6))

In [137]:
# y_train_residuals.plot(figsize=(24, 6))

In [138]:
## Uncomment code below if you want to have plots on feature importance.
## if one runs this after the previous code block, you're getting feature importance of the last step of walk-forward validation

# fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(10, 30))
# plot_importance(xgb_model, ax=ax1, importance_type='gain')
# ax1.set_title('Gain-based Importance', fontsize=12)

# plot_importance(xgb_model, ax=ax2, importance_type='weight')
# ax2.set_title('Split-based Importance', fontsize=12)

# plot_importance(xgb_model, ax=ax3, importance_type='cover')
# ax3.set_title('Cover Importance', fontsize=12)

# plot_importance(xgb_model, ax=ax4, importance_type='total_gain')
# ax4.set_title('Total Gain Importance', fontsize=12)

# plot_importance(xgb_model, ax=ax5, importance_type='total_cover')
# ax5.set_title('Total Cover Importance', fontsize=12)

## One Last Comparison

In [139]:
prophet_xgb_results_df

,fold,rmse,mape
0,0,14.571994,0.258205
1,1,8.007648,0.105853
2,2,6.605161,0.109265
3,3,12.470406,0.207969
4,4,14.457397,0.189021
5,5,18.959728,0.269195
6,6,13.324446,0.162700
7,7,10.980497,0.127049
8,8,12.508975,0.163871
9,9,12.372974,0.160191


In [140]:
prophet_results_df

,fold,rmse,mape
0,0,12.928150,0.245960
1,1,9.293687,0.142832
2,2,7.988090,0.130246
3,3,9.851386,0.143899
4,4,10.863706,0.137638
5,5,15.435334,0.215371
6,6,11.555594,0.145927
7,7,11.307567,0.135042
8,8,10.504938,0.117917
9,9,8.858023,0.110440


# Prophet Model for Daily Rat Sightings in New York City

For ease, we reimport the data. This way, we can run only the code in this section as opposed to having to run the code blocks above.

In [141]:
from prophet.plot import plot_plotly, plot_components_plotly

## Tuning Hyperparameters for Prophet

We make a copy of rs and set df as a copy of rs. This is so we don't mix up our variables. We will tune hyperparameters using df in the code below.

In [142]:
rs_saved = rs.copy()
df = rs.copy()

In [143]:
## Adding in holidays

date_range = pd.date_range(start="2020-01-01", end="2026-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

# Build the DataFrame in the same structure as your original
federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1,
})

holidays = federal_holidays

In [144]:
## To tune for hyperparameters, add more possible parameters to the dictionary below and add more values to it.
## So far, the I've been able to get is {'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5}

init_days = '2054 days'
cv_period = '14 days'
forecast_horizon = '14 days'

param_grid = {  
    'changepoint_prior_scale': [0.1, 1],
    'seasonality_prior_scale': [0.1, 1, 5, 10],
}

# Generate all combinations of parameters
all_params = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]
rmses = []  # Store the RMSEs for each params here
performance = []

# Use cross validation to evaluate all parameters
for params in all_params:
    params['holidays'] = holidays
    m = Prophet(**params).fit(df)  # Fit model with given params
    df_cv = cross_validation(m, initial = init_days, period=cv_period, horizon = forecast_horizon)
    df_p = performance_metrics(df_cv, rolling_window=14)
    performance.append(df_p)
    rmses.append(df_p['rmse'].values[0])

# Find the best parameters
tuning_results = pd.DataFrame(all_params)
tuning_results['rmse'] = rmses

best_params = all_params[np.argmin(rmses)]

23:30:49 - cmdstanpy - INFO - Chain [1] start processing
23:30:49 - cmdstanpy - INFO - Chain [1] done processing


  0%|          | 0/14 [00:00<?, ?it/s]

23:30:50 - cmdstanpy - INFO - Chain [1] start processing
23:30:50 - cmdstanpy - INFO - Chain [1] done processing
23:30:51 - cmdstanpy - INFO - Chain [1] start processing
23:30:51 - cmdstanpy - INFO - Chain [1] done processing
23:30:51 - cmdstanpy - INFO - Chain [1] start processing
23:30:52 - cmdstanpy - INFO - Chain [1] done processing
23:30:52 - cmdstanpy - INFO - Chain [1] start processing
23:30:52 - cmdstanpy - INFO - Chain [1] done processing
23:30:53 - cmdstanpy - INFO - Chain [1] start processing
23:30:53 - cmdstanpy - INFO - Chain [1] done processing
23:30:53 - cmdstanpy - INFO - Chain [1] start processing
23:30:54 - cmdstanpy - INFO - Chain [1] done processing
23:30:54 - cmdstanpy - INFO - Chain [1] start processing
23:30:55 - cmdstanpy - INFO - Chain [1] done processing
23:30:55 - cmdstanpy - INFO - Chain [1] start processing
23:30:55 - cmdstanpy - INFO - Chain [1] done processing
23:30:56 - cmdstanpy - INFO - Chain [1] start processing
23:30:56 - cmdstanpy - INFO - Chain [1]

  0%|          | 0/14 [00:00<?, ?it/s]

23:31:01 - cmdstanpy - INFO - Chain [1] start processing
23:31:01 - cmdstanpy - INFO - Chain [1] done processing
23:31:02 - cmdstanpy - INFO - Chain [1] start processing
23:31:02 - cmdstanpy - INFO - Chain [1] done processing
23:31:02 - cmdstanpy - INFO - Chain [1] start processing
23:31:03 - cmdstanpy - INFO - Chain [1] done processing
23:31:03 - cmdstanpy - INFO - Chain [1] start processing
23:31:03 - cmdstanpy - INFO - Chain [1] done processing
23:31:04 - cmdstanpy - INFO - Chain [1] start processing
23:31:04 - cmdstanpy - INFO - Chain [1] done processing
23:31:05 - cmdstanpy - INFO - Chain [1] start processing
23:31:05 - cmdstanpy - INFO - Chain [1] done processing
23:31:06 - cmdstanpy - INFO - Chain [1] start processing
23:31:06 - cmdstanpy - INFO - Chain [1] done processing
23:31:06 - cmdstanpy - INFO - Chain [1] start processing
23:31:07 - cmdstanpy - INFO - Chain [1] done processing
23:31:07 - cmdstanpy - INFO - Chain [1] start processing
23:31:07 - cmdstanpy - INFO - Chain [1]

...

23:31:55 - cmdstanpy - INFO - Chain [1] start processing
23:31:56 - cmdstanpy - INFO - Chain [1] done processing
23:31:57 - cmdstanpy - INFO - Chain [1] start processing
23:31:57 - cmdstanpy - INFO - Chain [1] done processing
23:31:58 - cmdstanpy - INFO - Chain [1] start processing
23:31:58 - cmdstanpy - INFO - Chain [1] done processing
23:31:59 - cmdstanpy - INFO - Chain [1] start processing
23:32:00 - cmdstanpy - INFO - Chain [1] done processing
23:32:00 - cmdstanpy - INFO - Chain [1] start processing
23:32:01 - cmdstanpy - INFO - Chain [1] done processing
23:32:01 - cmdstanpy - INFO - Chain [1] start processing
23:32:03 - cmdstanpy - INFO - Chain [1] done processing
23:32:03 - cmdstanpy - INFO - Chain [1] start processing
23:32:04 - cmdstanpy - INFO - Chain [1] done processing
23:32:04 - cmdstanpy - INFO - Chain [1] start processing
23:32:05 - cmdstanpy - INFO - Chain [1] done processing
23:32:06 - cmdstanpy - INFO - Chain [1] start processing
23:32:06 - cmdstanpy - INFO - Chain [1]

  0%|          | 0/14 [00:00<?, ?it/s]

23:32:14 - cmdstanpy - INFO - Chain [1] start processing
23:32:15 - cmdstanpy - INFO - Chain [1] done processing
23:32:15 - cmdstanpy - INFO - Chain [1] start processing
23:32:16 - cmdstanpy - INFO - Chain [1] done processing
23:32:16 - cmdstanpy - INFO - Chain [1] start processing
23:32:17 - cmdstanpy - INFO - Chain [1] done processing
23:32:18 - cmdstanpy - INFO - Chain [1] start processing
23:32:18 - cmdstanpy - INFO - Chain [1] done processing
23:32:19 - cmdstanpy - INFO - Chain [1] start processing
23:32:20 - cmdstanpy - INFO - Chain [1] done processing
23:32:20 - cmdstanpy - INFO - Chain [1] start processing
23:32:21 - cmdstanpy - INFO - Chain [1] done processing
23:32:22 - cmdstanpy - INFO - Chain [1] start processing
23:32:22 - cmdstanpy - INFO - Chain [1] done processing
23:32:23 - cmdstanpy - INFO - Chain [1] start processing
23:32:24 - cmdstanpy - INFO - Chain [1] done processing
23:32:24 - cmdstanpy - INFO - Chain [1] start processing
23:32:25 - cmdstanpy - INFO - Chain [1]

  0%|          | 0/14 [00:00<?, ?it/s]

23:32:32 - cmdstanpy - INFO - Chain [1] start processing
23:32:33 - cmdstanpy - INFO - Chain [1] done processing
23:32:34 - cmdstanpy - INFO - Chain [1] start processing
23:32:34 - cmdstanpy - INFO - Chain [1] done processing
23:32:35 - cmdstanpy - INFO - Chain [1] start processing
23:32:36 - cmdstanpy - INFO - Chain [1] done processing
23:32:36 - cmdstanpy - INFO - Chain [1] start processing
23:32:37 - cmdstanpy - INFO - Chain [1] done processing
23:32:37 - cmdstanpy - INFO - Chain [1] start processing
23:32:38 - cmdstanpy - INFO - Chain [1] done processing
23:32:39 - cmdstanpy - INFO - Chain [1] start processing
23:32:40 - cmdstanpy - INFO - Chain [1] done processing
23:32:40 - cmdstanpy - INFO - Chain [1] start processing
23:32:41 - cmdstanpy - INFO - Chain [1] done processing
23:32:41 - cmdstanpy - INFO - Chain [1] start processing
23:32:42 - cmdstanpy - INFO - Chain [1] done processing
23:32:43 - cmdstanpy - INFO - Chain [1] start processing
23:32:43 - cmdstanpy - INFO - Chain [1]

In [145]:
new_performance = pd.concat(performance, ignore_index=True)

# Round numeric columns for readability
numeric_cols = ["mse", "rmse", "mae", "mape", "mdape", "smape", "coverage"]
new_performance[numeric_cols] = new_performance[numeric_cols].round(4)


new_performance

,horizon,mse,rmse,mae,mape,mdape,smape,coverage
0,14 days,142.2601,11.9273,9.6061,0.3244,0.2428,0.3052,0.8776
1,14 days,141.3321,11.8883,9.5662,0.3234,0.2431,0.3053,0.8929
2,14 days,141.3546,11.8893,9.5800,0.3239,0.2432,0.3056,0.8673
3,14 days,141.1426,11.8803,9.5672,0.3234,0.2436,0.3045,0.8827
4,14 days,146.4931,12.1034,9.6572,0.3269,0.2385,0.3383,0.8520
5,14 days,145.7510,12.0727,9.6467,0.3264,0.2358,0.3389,0.8520
6,14 days,145.6748,12.0696,9.6284,0.3260,0.2361,0.3387,0.8571
7,14 days,146.1802,12.0905,9.6539,0.3266,0.2370,0.3392,0.8571


In [146]:
best_params

{'changepoint_prior_scale': 0.1,
 'seasonality_prior_scale': 10,
 'holidays':        holiday         ds  lower_window  upper_window
 0   federal_us 2020-01-01             0             1
 1   federal_us 2020-01-20             0             1
 2   federal_us 2020-02-17             0             1
 3   federal_us 2020-05-25             0             1
 4   federal_us 2020-07-03             0             1
 ..         ...        ...           ...           ...
 63  federal_us 2025-11-27             0             1
 64  federal_us 2025-12-25             0             1
 65  federal_us 2026-01-01             0             1
 66  federal_us 2026-01-19             0             1
 67  federal_us 2026-02-16             0             1
 
 [68 rows x 4 columns]}

In [147]:
m = Prophet(**best_params)
m.add_country_holidays(country_name='US')
m.fit(df)
future = m.make_future_dataframe(periods=14)
forecast = m.predict(future)

23:32:50 - cmdstanpy - INFO - Chain [1] start processing
23:32:50 - cmdstanpy - INFO - Chain [1] done processing


In [148]:
fig1 = plot_plotly(m, forecast)
fig1.show()

fig2 = plot_components_plotly(m, forecast)
fig2.show()